# Left-Wing Non-Fiction Books — Open Library API
Fetches books by selected left-wing authors and saves results to `Data/Raw/non fiction/leftpolitics_raw.csv`.

In [ ]:
import requests
import pandas as pd
import time

In [ ]:
AUTHORS = [
    # Already included
    "Angela Davis",
    "James Baldwin",
    "Mahmoud Darwish",
    "Noam Chomsky",
    "Naomi Klein",
    "bell hooks",
    "Howard Zinn",
    "Eduardo Galeano",
    "Frantz Fanon",
    "Rosa Luxemburg",
    "Antonio Gramsci",
    "Michel Foucault",
    "Cornel West",
    "Ta-Nehisi Coates",
    "Robin D.G. Kelley",

    # Black American & diaspora thinkers
    "W.E.B. Du Bois",
    "Audre Lorde",
    "Malcolm X",
    "Cedric Robinson",
    "Saidiya Hartman",
    "Keeanga-Yamahtta Taylor",
    "Patricia Hill Collins",
    "Manning Marable",
    "Grace Lee Boggs",
    "Ibram X. Kendi",

    # African & Caribbean theorists
    "C.L.R. James",
    "Aimé Césaire",
    "Walter Rodney",
    "Achille Mbembe",
    "Stuart Hall",

    # Latin American & Global South
    "Paulo Freire",
    "Vijay Prashad",
    "José Carlos Mariátegui",

    # Feminist & queer theory
    "Sara Ahmed",
    "Silvia Federici",
    "Gayatri Chakravorty Spivak",

    # Contemporary left
    "David Graeber",
    "Mark Fisher",
    "Slavoj Žižek",
]

In [ ]:
SEARCH_URL = "https://openlibrary.org/search.json"


def fetch_books_for_author(author_name, max_results=50):
    params = {
        "author": author_name,
        "fields": "title,author_name,first_publish_year,subject,edition_count,key",
        "limit": max_results,
    }
    try:
        resp = requests.get(SEARCH_URL, params=params, timeout=15)
        resp.raise_for_status()
        docs = resp.json().get("docs", [])
    except requests.RequestException as e:
        print(f"  Error fetching '{author_name}': {e}")
        return []

    records = []
    for doc in docs:
        authors = doc.get("author_name", [])
        if not any(author_name.lower() in a.lower() for a in authors):
            continue
        subjects = doc.get("subject", [])
        records.append({
            "title": doc.get("title", ""),
            "author": ", ".join(authors),
            "year_published": doc.get("first_publish_year"),
            "subjects": " | ".join(subjects[:10]) if subjects else "",
            "edition_count": doc.get("edition_count", 0),
            "open_library_key": doc.get("key", ""),
            "queried_author": author_name,
        })
    return records

In [ ]:
all_records = []

for author in AUTHORS:
    print(f"Fetching: {author}...")
    books = fetch_books_for_author(author)
    print(f"  → {len(books)} books found")
    all_records.extend(books)
    time.sleep(0.5)

print(f"\nTotal records collected: {len(all_records)}")

In [ ]:
df = pd.DataFrame(all_records)

df = df.drop_duplicates(subset=["title", "author"])
df = df.sort_values(["queried_author", "year_published"], na_position="last")
df = df.reset_index(drop=True)

print(f"Unique records after deduplication: {len(df)}")
df.head(10)

In [ ]:
import os

output_dir = "Data/Raw/non fiction"
output_path = os.path.join(output_dir, "leftpolitics_raw.csv")

os.makedirs(output_dir, exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} rows to {output_path}")